In [40]:
import gzip
import shutil
import pandas as pd
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import spacy
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import pyLDAvis
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel

In [5]:
# Скачиваем стоп-слова
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
!pip install -q spacy
!python -m spacy download ru_core_news_sm
nlp = spacy.load('ru_core_news_sm')

     ---------------------------------------- 0.0/15.3 MB ? eta -:--:--
     - -------------------------------------- 0.5/15.3 MB 5.7 MB/s eta 0:00:03
     -- ------------------------------------- 1.0/15.3 MB 3.0 MB/s eta 0:00:05
     ---- ----------------------------------- 1.8/15.3 MB 3.6 MB/s eta 0:00:04
     ----- ---------------------------------- 2.1/15.3 MB 3.1 MB/s eta 0:00:05
     ------- -------------------------------- 2.9/15.3 MB 2.9 MB/s eta 0:00:05
     -------- ------------------------------- 3.4/15.3 MB 2.9 MB/s eta 0:00:05
     ---------- ----------------------------- 3.9/15.3 MB 2.8 MB/s eta 0:00:05
     ----------- ---------------------------- 4.5/15.3 MB 2.8 MB/s eta 0:00:04
     ------------ --------------------------- 4.7/15.3 MB 2.7 MB/s eta 0:00:04
     -------------- ------------------------- 5.5/15.3 MB 2.7 MB/s eta 0:00:04
     --------------- ------------------------ 6.0/15.3 MB 2.7 MB/s eta 0:00:04
     ----------------- ---------------------- 6.6/15.3 MB 2

In [10]:
# Скачиваем архив с новостями Lenta.Ru
import urllib.request

# URL архива с новостями Lenta.Ru
url = "https://github.com/yutkin/Lenta.Ru-News-Dataset/releases/download/v1.0/lenta-ru-news.csv.gz"
# Путь для сохранения файла
output_path = "lenta-ru-news.csv.gz"
# Загружаем файл
urllib.request.urlretrieve(url, output_path)

('lenta-ru-news.csv.gz', <http.client.HTTPMessage at 0x2ace4cbdba0>)

In [12]:
input_path = 'lenta-ru-news.csv.gz'  # Путь к сжатому файлу (должен совпадать с output_path выше)
output_csv_path = 'lenta-ru-news.csv'  # Путь для распакованного CSV файла
with gzip.open(input_path, 'rb') as f_in:  # Открываем gzip-архив в режиме чтения бинарных данных
    with open(output_csv_path, 'wb') as f_out:  # Открываем файл для записи распакованных данных
        shutil.copyfileobj(f_in, f_out)  # Копируем содержимое архива в новый файл, распаковывая его

In [13]:
df = pd.read_csv(output_csv_path)  
print("Количество новостей:", len(df))

Количество новостей: 739351


In [14]:
# Для быстроты возьмем первые 1000 текстов
texts = df['text'][:1000].astype(str).tolist()

In [15]:
stop_words = set(stopwords.words('russian'))

In [ ]:
def preprocess(text):  
    doc = nlp(text.lower())
    tokens = [token.lemma_ for token in doc 
              if token.lemma_ not in stop_words 
              and not token.is_punct 
              and not token.is_space 
              and len(token.lemma_) > 2] 
    return ' '.join(tokens)

In [17]:
processed_texts = [preprocess(text) for text in texts]

In [56]:
vectorizer = CountVectorizer(min_df=5, max_df=0.9)  # Определяем vectorizer
data_vectorized = vectorizer.fit_transform(processed_texts)

In [19]:
num_topics = 10  # Задаём количество тем для модели LDA
lda_model = LatentDirichletAllocation(n_components=num_topics,  # Создаём модель LDA с указанным числом тем
                                      max_iter=20,  # Устанавливаем максимальное количество итераций
                                      learning_method='batch',  # Используем метод обучения "batch" (все данные за раз)
                                      random_state=42)  # Фиксируем начальное состояние для воспроизводимости
lda_model.fit(data_vectorized)  # Обучаем модель на векторизованных текстах

,n_components,10
,doc_topic_prior,None
,topic_word_prior,None
,learning_method,'batch'
,learning_decay,0.7
,learning_offset,10.0
,max_iter,20
,batch_size,128
,evaluate_every,-1
,total_samples,1000000.0
,perp_tol,0.1


In [57]:
def print_topics(model, vectorizer, top_n=15):  # Определяем функцию для вывода ключевых слов тем
    for idx, topic in enumerate(model.components_):  # Перебираем компоненты модели (темы)
        print(f"\nТема {idx + 1}:")  # Выводим номер темы
        top_features = [vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-top_n - 1:-1]]  # Извлекаем топ-N слов для темы
        print(", ".join(top_features))  # Выводим слова, объединённые запятыми

print_topics(lda_model, count_vectorizer)


Тема 1:
год, это, суд, убийство, человек, журналист, декабрь, ребёнок, саудовский, смерть, сообщать, город, свой, тысяча, обнаружить

Тема 2:
это, церковь, декабрь, свой, слово, российский, украина, глава, православный, собор, заявить, рассказать, сказать, также, упц

Тема 3:
страна, это, россия, также, декабрь, заявить, франция, министр, правительство, мочь, документ, год, гражданин, свой, сообщать

Тема 4:
год, стать, пользователь, это, свой, опубликовать, также, 2018, первый, фильм, человек, декабрь, видео, фотография, весь

Тема 5:
год, рубль, это, процент, россия, миллион, компания, тысяча, цена, квартира, 2018, 2019, российский, миллиард, доллар

Тема 6:
процент, президент, путин, россия, владимир, год, глава, это, человек, государство, выбор, республика, развитие, страна, также

Тема 7:
самолёт, год, российский, это, декабрь, россия, время, система, также, военный, авиакомпания, компания, ракета, комплекс, работа

Тема 8:
сша, это, год, свой, американский, россия, российский, з

In [43]:
import pyLDAvis
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel

In [58]:
# Визуализация: Гистограмма распределения тем
def plot_topic_distribution(lda_model, data_vectorized):
    topic_distributions = lda_model.transform(data_vectorized)
    print("Размер topic_distributions:", topic_distributions.shape)
    topic_sums = np.sum(topic_distributions, axis=0)
    print("Суммы по темам:", topic_sums)
    topic_sums = topic_sums / np.sum(topic_sums) * 100
    print("Нормализованные суммы (%):", topic_sums)
    plt.figure(figsize=(10, 6))
    plt.bar(range(1, len(topic_sums) + 1), topic_sums, color='skyblue', edgecolor='black')
    plt.xlabel('Тема')
    plt.ylabel('Доля в наборе данных (%)')
    plt.title('Распределение тем по документам')
    plt.xticks(range(1, len(topic_sums) + 1))
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig('topic_distribution.png')
    print("График сохранён в 'topic_distribution.png'")
    plt.show()

In [59]:
# Визуализация: Облака слов
def plot_word_clouds(lda_model, vectorizer, num_topics):
    feature_names = vectorizer.get_feature_names_out()
    for topic_idx, topic in enumerate(lda_model.components_):
        word_weights = {feature_names[i]: topic[i] for i in topic.argsort()[:-50 - 1:-1]}
        wordcloud = WordCloud(width=800, height=400, background_color='white',
                             colormap='viridis', max_words=50).generate_from_frequencies(word_weights)
        plt.figure(figsize=(8, 4))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'Тема {topic_idx + 1}')
        plt.tight_layout()
        plt.savefig(f'wordcloud_topic_{topic_idx + 1}.png')
        print(f"Облако слов для темы {topic_idx + 1} сохранено в 'wordcloud_topic_{topic_idx + 1}.png'")
        plt.show()

In [60]:
panel = pyLDAvis.prepare(lda_model.components_, data_vectorized, count_vectorizer, mds='tsne')
pyLDAvis.save_html(panel, 'lda_visualization.html')
print("Интерактивная визуализация сохранена в 'lda_visualization.html'")

TypeError: prepare() missing 2 required positional arguments: 'vocab' and 'term_frequency'

In [61]:
perplexity = lda_model.perplexity(data_vectorized)
print(f'Перплексия: {perplexity}')

Перплексия: 1891.1428340320003


In [62]:
texts_tokenized = [text.split() for text in processed_texts]
dictionary = Dictionary(texts_tokenized)
corpus = [dictionary.doc2bow(text) for text in texts_tokenized]

In [63]:
# Вычисляем когерентность (u_mass) для текущей модели
topics = [[(vectorizer.get_feature_names_out()[i], topic[i]) for i in topic.argsort()[:-50 - 1:-1]]
          for topic in lda_model.components_]
coherence_model_lda = CoherenceModel(topics=topics, corpus=corpus, dictionary=dictionary, coherence='u_mass')
coherence_lda = coherence_model_lda.get_coherence()
print('U-Mass:', coherence_lda)

ValueError: unable to interpret topic as either a list of tokens or a list of ids